# 🤖 RPA em Python — Extração de Clientes e Envio ao Webhook n8n

Este notebook implementa a etapa de **RPA (Robotic Process Automation)** do desafio **Criando um Assistente de Investimentos com RPA e IA Generativa** da DIO.

### O que foi completado em relação ao notebook base da DIO:
1. Extração automatizada das 10 linhas da tabela HTML (`#clientes tbody tr`) com `requests` e `BeautifulSoup`.
2. Implementação completa do bloco **`TODO (N8N)`** para envio do JSON `{"clientes": clientes}` via `POST` ao nó `Webhook` (`/Clientes`) do n8n com tratamento de exceções e timeout.

In [1]:
!pip install requests beautifulsoup4

In [2]:
import json
import os
import requests
from bs4 import BeautifulSoup

URL_PAGINA = "https://digitalinnovationone.github.io/dio-lab-assistente-investimentos-rpa-n8n/"

# 1) Baixar a página HTML de clientes
response = requests.get(URL_PAGINA, timeout=15)
response.raise_for_status()
response.encoding = "utf-8"
html = response.text

# 2) Ler a tabela de clientes (#clientes tbody tr)
soup = BeautifulSoup(html, "html.parser")
linhas = soup.select("#clientes tbody tr")

clientes = []

for linha in linhas:
    colunas = linha.find_all("td")
    cliente = {
        "nome": colunas[0].get_text(strip=True),
        "email": colunas[1].get_text(strip=True),
        "saldo": colunas[2].get_text(strip=True),
        "perfil": colunas[3].get_text(strip=True),
    }
    clientes.append(cliente)

# 3) Mostrar o resultado extraído
print(f"Clientes encontrados ({len(clientes)}):")
for cliente in clientes:
    print(cliente)

# 4) Enviar a lista 'clientes' para o Webhook do n8n via POST (TODO resolvido)
N8N_WEBHOOK_URL = os.environ.get("N8N_WEBHOOK_URL", "http://localhost:5678/webhook-test/Clientes")
payload = {"clientes": clientes}

try:
    r_n8n = requests.post(N8N_WEBHOOK_URL, json=payload, timeout=30)
    r_n8n.raise_for_status()
    print("\nResposta do n8n:")
    print(json.dumps(r_n8n.json(), ensure_ascii=False, indent=2))
except Exception as err:
    print(f"\n[Aviso] Webhook n8n não acessível no momento ({err}).")
    print("Payload JSON pronto para envio:")
    print(json.dumps(payload, ensure_ascii=False, indent=2))
